# 12 — The combo: dynamic topology + dynamic agents + dynamic tools

**Goal:** put **all three** autonomy features together and watch a real coding benchmark improve.

Throughout these notebooks we have built three independent capabilities:

| Capability | What it does | Where |
|---|---|---|
| **Dynamic topology** | An agent picks the shape of the workflow per request | Notebook 10 |
| **Dynamic agents** | Spawn agents from declarative `AgentSpec` records | Notebook 02 |
| **Dynamic tools** | Generate tool code at runtime and run it in a sandbox | Notebook 11 |

This notebook combines all three on a real task: **solving coding problems with a test-driven loop**.

The architecture:

1. A **designer agent** sees the problem and picks a topology shape (single agent, pipeline, or refinement loop).
2. **Coder** and **fixer** agents are spawned from `AgentSpec` records — no Python subclassing.
3. When the design needs verification, a fresh **`GeneratedToolSpec`** is created per iteration per test. Each runs the candidate code in the sandbox.
4. The fixer sees which tests failed and revises. The loop keeps the **best candidate so far** (never regress on visible tests).

**The win** (averaged over 3 trials on `gpt-4o-mini`):

| Metric | Baseline | Combo | Δ |
|---|---|---|---|
| `pass@1` | 73% (±6%) | **90%** (±10%) | **+17pp** |
| `test_pass_rate` | 80% (±6%) | **94%** (±9%) | **+14pp** |

Highlights per problem: `parse_csv_row` 23% → 73%, `word_ladder_length` 38% → 100%, `parse_roman` 55% → 79%. **Zero regressions.**

**The honest framing.** The combo wins not because the model is better, but because the system **runs its own candidate code against examples** and iterates. A single LLM cannot always anticipate edge cases in one shot. Self-testing closes that gap.

**Cost.** This notebook makes ~15–20 real LLM calls (~$0.02 on `gpt-4o-mini`). The 3-trial benchmark above is run separately via `benchmarks/coding/run.py` and the results are loaded into the final cell.


## 1. Setup

In [1]:
from __future__ import annotations
import asyncio
import time
from typing import Any
from dataclasses import dataclass

from pydantic import BaseModel, Field

from orqest import load_config
from orqest.agents import BaseAgent, GlobalState
from orqest.autonomy import (
    AgentFactory, AgentSpec, ToolRegistry,
    DynamicToolFactory, GeneratedToolSpec,
)
from orqest.sandbox import SubprocessSandbox

config = load_config()
print(f"Model: {config.llm_model}")

Model: openai:gpt-5.2


## 2. The mini-CodeBench

A coding benchmark with two genuinely tricky problems:

- **`parse_roman`** — strict Roman numeral validation. The catch: `"IIII"` is invalid Roman, even though most LLMs accept it on the first try.
- **`parse_csv_row`** — RFC 4180 CSV parsing, including the rule that a double quote inside a quoted field is escaped by doubling it.

Each problem has a `prompt` (what the agent sees) and a list of `tests` (hidden checks). Some tests are shown to the agent as **visible examples**. The rest are kept hidden for scoring.

The benchmark loaded here is small (2 problems) so the notebook runs fast. The full 10-problem benchmark lives in `benchmarks/coding/run.py`.


In [2]:
@dataclass(frozen=True)
class Problem:
    name: str
    prompt: str
    tests: list[tuple[str, Any]]


PROBLEMS = [
    Problem(
        name="parse_roman",
        prompt=(
            "Implement `parse_roman(s: str) -> int | None`. "
            "Convert a Roman numeral string to integer for 1..3999. "
            "STRICT validation: return None for any invalid Roman numeral. "
            "Rules: I, X, C can appear up to 3 times in a row; V, L, D appear "
            "at most once total; only IV/IX, XL/XC, CD/CM are valid subtractive forms. "
            "'IIII' is invalid (return None). 'VV' is invalid. 'IL' is invalid. "
            "Empty string returns None. Lowercase or mixed case returns None. "
            "Return ONLY the function definition."
        ),
        tests=[
            ("parse_roman('III')", 3),
            ("parse_roman('IV')", 4),
            ("parse_roman('IX')", 9),
            ("parse_roman('LVIII')", 58),
            ("parse_roman('MCMXCIV')", 1994),
            ("parse_roman('MMMCMXCIX')", 3999),
            ("parse_roman('IIII')", None),
            ("parse_roman('VV')", None),
            ("parse_roman('IL')", None),
            ("parse_roman('IC')", None),
            ("parse_roman('')", None),
            ("parse_roman('iv')", None),
            ("parse_roman('XIIII')", None),
            ("parse_roman('VIII')", 8),
        ],
    ),
    Problem(
        name="parse_csv_row",
        prompt=(
            "Implement `parse_csv_row(line: str) -> list[str]`. "
            "Parse ONE CSV line according to RFC 4180 rules: "
            "- Fields are separated by commas. "
            "- A field may be quoted with double quotes; quoted fields may contain commas. "
            "- Inside a quoted field, a literal double quote is escaped by doubling it: \"\". "
            "- Unquoted fields contain no quotes. "
            "- Empty fields are preserved (a trailing comma produces a trailing empty string). "
            "Return ONLY the function definition."
        ),
        tests=[
            ("parse_csv_row('a,b,c')", ["a", "b", "c"]),
            ("parse_csv_row('\"a\",\"b\",\"c\"')", ["a", "b", "c"]),
            ("parse_csv_row('a,\"b,c\",d')", ["a", "b,c", "d"]),
            ("parse_csv_row('\"a\"\"b\"')", ['a\"b']),
            ("parse_csv_row('a,,c')", ["a", "", "c"]),
            ("parse_csv_row('a,b,')", ["a", "b", ""]),
            ("parse_csv_row(',a')", ["", "a"]),
            ("parse_csv_row('')", [""]),
            ("parse_csv_row('\"hello, world\",foo')", ["hello, world", "foo"]),
            ("parse_csv_row('a,\"b\"\"c\"\"d\",e')", ["a", 'b\"c\"d', "e"]),
        ],
    ),
]


def visible_examples(p: Problem, k: int = 4) -> list[tuple[str, Any]]:
    return p.tests[:k]


def score_candidate(code_str: str, p: Problem) -> dict:
    """Run candidate code in-process and grade against every test."""
    result = {"problem": p.name, "passed": 0, "total": len(p.tests), "errors": []}
    ns: dict = {}
    try:
        exec(code_str, ns)
    except Exception as exc:
        result["compile_error"] = f"{type(exc).__name__}: {exc}"
        return result
    for expr, expected in p.tests:
        try:
            actual = eval(expr, ns)
            if actual == expected:
                result["passed"] += 1
            else:
                result["errors"].append({"expr": expr, "expected": expected, "actual": actual})
        except Exception as exc:
            result["errors"].append({"expr": expr, "expected": expected,
                                     "actual": f"<{type(exc).__name__}: {exc}>"})
    return result


print(f"{len(PROBLEMS)} problems, "
      f"{sum(len(p.tests) for p in PROBLEMS)} hidden tests total.\n")
for p in PROBLEMS:
    print(f"  {p.name:20s} {len(p.tests)} tests, first visible: {p.tests[0][0]}")

2 problems, 24 hidden tests total.

  parse_roman          14 tests, first visible: parse_roman('III')
  parse_csv_row        10 tests, first visible: parse_csv_row('a,b,c')


## 3. Baseline — single-shot agent

The simplest possible architecture: one agent, one LLM call, one answer. The agent sees the problem and a few visible examples, and emits a `CodeSolution` in one shot.

This is what most "let an LLM write code" systems do. The benchmark score from this baseline is what the combo has to beat.


In [3]:
class CodeSolution(BaseModel):
    reasoning: str = Field(description="Short reasoning (one or two sentences).")
    code: str = Field(description="Complete Python function definition. No example calls, no markdown.")


class BaselineCoder(BaseAgent[GlobalState, CodeSolution]):
    def __init__(self, model: str, api_key: str):
        super().__init__(
            agent_name="baseline_coder",
            system_prompt=(
                "You are an expert Python programmer. Read the problem carefully and write "
                "a correct function definition. Pay attention to edge cases mentioned in the "
                "prompt. Return ONLY the function definition. Available stdlib: re, math, "
                "collections, itertools, string, bisect, heapq, functools, operator."
            ),
            output_type=CodeSolution,
            model=model,
            api_key=api_key,
            retries=2,
        )

    async def _run_implementation(self, state, **kwargs) -> CodeSolution:
        result = await self.call_model(state.get_latest_message("user"), state)
        return result.output


async def run_baseline(problem: Problem, agent: BaselineCoder, k: int = 4) -> dict:
    visible = visible_examples(problem, k=k)
    state = GlobalState()
    state.add_message(
        "user",
        f"PROBLEM:\n{problem.prompt}\n\nVisible examples:\n"
        + "\n".join(f"  {e} == {x!r}" for e, x in visible),
    )
    sol = await agent.run(state)
    score = score_candidate(sol.code, problem)
    score["code"] = sol.code
    return score


baseline = BaselineCoder(model=config.llm_model, api_key=config.llm_api_key)
parse_roman = PROBLEMS[0]
result = await run_baseline(parse_roman, baseline)

print(f"baseline on {parse_roman.name}: {result['passed']}/{result['total']} tests pass")
if result["errors"]:
    print(f"\nFirst failed test:")
    e = result["errors"][0]
    print(f"  {e['expr']}")
    print(f"  expected: {e['expected']!r}")
    print(f"  got:      {e['actual']!r}")

baseline on parse_roman: 14/14 tests pass


## 4. Pillar 1 — topology design

The `DesignerAgent` reads the problem and picks a topology shape. For verifier-decidable problems with subtle edge cases, it picks `refinement_pipeline` — a coder followed by an iterative fixer. For simpler problems it picks `pipeline` or `single`.

Same agent class — **different topologies emerge per problem**. The output is a typed Pydantic model, so the framework can validate it, inspect it, and (in production) cache it.


In [4]:
class TopologyDesign(BaseModel):
    """The shape the designer picked for the goal."""
    thought: str = Field(description="Why this shape fits the goal.")
    shape: str = Field(description="One of: 'single', 'pipeline', 'refinement_pipeline'.")
    needs_iteration: bool = Field(description="True if test-driven refinement is recommended.")
    coder_role: str = Field(description="One-sentence description of the coder agent's job.")
    fixer_role: str | None = Field(default=None, description="Fixer's job, if iteration needed.")
    max_iterations: int = Field(default=3)


class DesignerAgent(BaseAgent[GlobalState, TopologyDesign]):
    def __init__(self, model: str, api_key: str):
        super().__init__(
            agent_name="designer",
            system_prompt=(
                "You are a multi-agent system architect. Given a problem statement, decide "
                "which topology fits best from these options: "
                "'single' (one shot, for easy problems), "
                "'pipeline' (multi-step but no iteration, for problems with clear stages), "
                "'refinement_pipeline' (coder + iterative fixer with sandbox-based test "
                "execution, for problems with subtle edge cases or hidden complexity). "
                "For coding tasks with hidden tests, prefer 'refinement_pipeline'. "
                "Always articulate WHY (1-2 sentences) and the agent roles."
            ),
            output_type=TopologyDesign,
            model=config.llm_model,
            api_key=config.llm_api_key,
            retries=2,
        )

    async def _run_implementation(self, state, **kwargs) -> TopologyDesign:
        result = await self.call_model(state.get_latest_message("user"), state)
        return result.output


designer = DesignerAgent(model=config.llm_model, api_key=config.llm_api_key)
state = GlobalState()
state.add_message(
    "user",
    f"PROBLEM:\n{parse_roman.prompt}\n\nDecide topology shape and articulate why.",
)
design = await designer._run_implementation(state)

print(f"shape:           {design.shape}")
print(f"needs_iteration: {design.needs_iteration}")
print(f"max_iterations:  {design.max_iterations}")
print(f"\nthought:\n  {design.thought}")
print(f"\ncoder_role:\n  {design.coder_role}")
if design.fixer_role:
    print(f"\nfixer_role:\n  {design.fixer_role}")

shape:           refinement_pipeline
needs_iteration: True
max_iterations:  3

thought:
  This is a coding task with strict validation rules and many subtle edge cases (repetition limits, subtractive constraints, case handling, empty input) that are easy to get wrong without tests, so iterative refinement with test-driven fixes is best.

coder_role:
  Implement parse_roman with strict Roman numeral validation and conversion, along with an initial suite of unit tests covering standard and edge cases.

fixer_role:
  Run hidden/edge-case tests (or reason through failures) and iteratively adjust validation logic until all cases pass while preserving strict rules.


## 5. Pillar 2 — dynamic agent spawning

Instead of writing `class CoderAgent(BaseAgent): ...` and `class FixerAgent(BaseAgent): ...` as Python subclasses, we declare them as `AgentSpec` records. `AgentFactory.spawn()` builds the live agent at runtime: it constructs the Pydantic output model from the JSON schema and wires up any tools.

Why this matters: an LLM (or an optimizer like GEPA from notebook 07) can **mutate** these specs without recompiling Python. You can persist them to disk and transmit them over the wire.


In [5]:
CODER_SPEC = AgentSpec(
    name="coder",
    system_prompt=(
        "You are an expert Python programmer in a test-driven loop. Write a clean "
        "first draft of the requested function. Your draft will be tested against the "
        "provided examples; if any fail, a fixer will revise it. Return ONLY the function."
    ),
    output_schema={
        "type": "object",
        "properties": {
            "reasoning": {"type": "string", "description": "Short reasoning."},
            "code": {"type": "string", "description": "Complete Python function definition."},
        },
        "required": ["reasoning", "code"],
    },
    tools=[],
    model=config.llm_model,
)

FIXER_SPEC = AgentSpec(
    name="fixer",
    system_prompt=(
        "You are a careful debugger. You see the previous code and which test cases "
        "passed vs failed (with expected and actual outputs). Diagnose the bug and "
        "emit a corrected COMPLETE function definition. PRESERVE everything that "
        "currently works — only change what is needed to fix the failing tests."
    ),
    output_schema={
        "type": "object",
        "properties": {
            "reasoning": {"type": "string", "description": "Diagnosis of the bug."},
            "code": {"type": "string", "description": "Complete corrected function definition."},
        },
        "required": ["reasoning", "code"],
    },
    tools=[],
    model=config.llm_model,
)


sandbox = SubprocessSandbox()
tool_factory = DynamicToolFactory(sandbox=sandbox)
agent_factory = AgentFactory(
    registry=ToolRegistry(),
    default_model=config.llm_model,
    api_key=config.llm_api_key,
    tool_factory=tool_factory,
)

coder = agent_factory.spawn(CODER_SPEC)
fixer = agent_factory.spawn(FIXER_SPEC)

print(f"spawned coder: {type(coder).__name__}('{coder.agent_name}')")
print(f"  output type: {coder.output_type.__name__}")
print(f"  output schema fields: {list(coder.output_type.model_fields)}")
print(f"\nspawned fixer: {type(fixer).__name__}('{fixer.agent_name}')")

spawned coder: DynamicAgent('coder')
  output type: coder_Output
  output schema fields: ['reasoning', 'code']

spawned fixer: DynamicAgent('fixer')


## 6. Pillar 3 — dynamic tool spawning

The refinement loop needs to actually **run** candidate Python code to know which tests pass. We do this with a `GeneratedToolSpec`: a tool whose body is the candidate code, with a one-liner at the end that calls the function and returns the result.

A **fresh tool is spawned per (iteration × test)**. Each tool runs once in the sandbox, returns the actual value, and is discarded. Truly dynamic — exactly the kind of work the sandbox was built for.

Let's prove it works. We build a deliberately **buggy** Roman numeral parser (one that accepts `"IIII"`) and use a `GeneratedToolSpec` to detect the bug.


In [6]:
ALLOWED_IMPORTS = {
    "re", "math", "collections", "itertools", "string",
    "bisect", "heapq", "functools", "operator",
}


# A deliberately buggy parse_roman — accepts "IIII".
buggy_candidate = """
def parse_roman(s):
    if not s:
        return None
    vals = {'I':1, 'V':5, 'X':10, 'L':50, 'C':100, 'D':500, 'M':1000}
    if any(ch not in vals for ch in s):
        return None
    total = 0
    prev = 0
    for ch in reversed(s):
        v = vals[ch]
        if v < prev:
            total -= v
        else:
            total += v
        prev = v
    return total
"""


# Build a per-test tool spec — implementation = candidate + return call.
test_expression = "parse_roman('IIII')"        # should be None per the spec
spec = GeneratedToolSpec(
    name="run_parse_roman_test",
    description=f"Invoke candidate to compute {test_expression}",
    implementation=f"{buggy_candidate}\nreturn {test_expression}\n",
    allowed_imports=ALLOWED_IMPORTS,
    timeout_s=4.0,
)

print(f"GeneratedToolSpec: {spec.name}")
print(f"  allowed_imports:  {sorted(spec.allowed_imports)}")
print(f"  timeout_s:        {spec.timeout_s}")
print(f"  impl length:      {len(spec.implementation)} chars")


# Spawn the tool through the factory and call it.
tool = await tool_factory.spawn(spec)
print(f"\nSpawned tool: {tool.name}")

actual = await tool.function()
expected = None                                # spec says 'IIII' is invalid
print(f"\nTest: {test_expression}")
print(f"  expected: {expected!r}")
print(f"  actual:   {actual!r}")
print(f"  → buggy candidate: {'CAUGHT' if actual != expected else 'PASSED (false positive)'}")

GeneratedToolSpec: run_parse_roman_test
  allowed_imports:  ['bisect', 'collections', 'functools', 'heapq', 'itertools', 'math', 'operator', 're', 'string']
  timeout_s:        4.0
  impl length:      403 chars

Spawned tool: run_parse_roman_test

Test: parse_roman('IIII')
  expected: None
  actual:   4
  → buggy candidate: CAUGHT


## 7. The combo loop — end-to-end on `parse_roman`

Now we wire everything together. The flow for one problem:

1. **Design** — designer picks the topology shape.
2. **Coder draft** — the coder produces the first version.
3. **Test** — every visible test runs in the sandbox. Get the pass count.
4. **Fix** — if any visible test failed and we have iterations left, the fixer sees the failures and revises.
5. **Keep best** — track the candidate with the most visible passes. Never regress.
6. **Score** — final code is graded against **all** tests (visible + hidden).


### 7a. Helpers — `evaluate` and `format_feedback`

`evaluate` runs the candidate against a list of tests, spawning a fresh `GeneratedToolSpec` per test. `format_feedback` builds the feedback message the fixer sees.


In [ ]:
async def evaluate(code_str: str, tests, tool_factory):
    """Run candidate against each test by spawning a fresh GeneratedToolSpec per call."""
    n_pass = 0
    records = []
    for idx, (expr, expected) in enumerate(tests):
        spec = GeneratedToolSpec(
            name=f"candidate_{idx}",
            description=f"Invoke candidate for {expr}",
            implementation=f"{code_str}\nreturn {expr}\n",
            allowed_imports=ALLOWED_IMPORTS,
            timeout_s=4.0,
        )
        try:
            tool = await tool_factory.spawn(spec)
            actual = await tool.function()
        except Exception as exc:
            records.append({"expr": expr, "passed": False,
                            "actual": f"<{type(exc).__name__}>",
                            "expected": repr(expected)[:60]})
            continue
        if isinstance(actual, dict) and actual.get("stage") == "sandbox.execute":
            records.append({"expr": expr, "passed": False,
                            "actual": "<sandbox error>", "expected": repr(expected)[:60]})
            continue
        passed = actual == expected
        if passed:
            n_pass += 1
        records.append({"expr": expr, "passed": passed,
                        "actual": repr(actual)[:60], "expected": repr(expected)[:60]})
    return n_pass, records


def format_feedback(records):
    lines = []
    for r in records:
        marker = "PASS" if r["passed"] else "FAIL"
        lines.append(f"  {marker}: {r['expr']}")
        if not r["passed"]:
            lines.append(f"         expected {r['expected']}")
            lines.append(f"         got      {r['actual']}")
    return "\n".join(lines)


print("evaluate and format_feedback ready")

### 7b. The main loop

`run_combo` is the entry point. It does design → coder draft → optional refinement loop → keep best.

Notice the **keep-best** policy: we only adopt a fixer's revision if it improves the visible pass count. This guarantees the loop never regresses.


In [7]:
async def run_combo(problem, designer, coder, fixer, tool_factory, k_visible=4, max_iter=3):
    """The combo: designer → coder → refinement loop with sandbox tests."""
    visible = visible_examples(problem, k=k_visible)

    # Step 1: designer picks the shape.
    d_state = GlobalState()
    d_state.add_message("user",
        f"PROBLEM:\n{problem.prompt}\n\nVisible examples:\n"
        + "\n".join(f"  {e} == {x!r}" for e, x in visible)
        + "\n\nDecide topology shape."
    )
    design = await designer._run_implementation(d_state)
    print(f"  ↪ designer picked: {design.shape}")

    # Step 2: coder draft.
    c_state = GlobalState()
    c_state.add_message("user",
        f"PROBLEM:\n{problem.prompt}\n\nVisible examples:\n"
        + "\n".join(f"  {e} == {x!r}" for e, x in visible))
    initial = await coder.run(c_state)
    best_code = initial.code
    best_pass, records = await evaluate(best_code, visible, tool_factory)
    print(f"  ↪ iter 0 (coder draft): {best_pass}/{len(visible)} visible tests pass")

    current_code = best_code
    current_records = records

    # Step 3: refinement loop, only if the designer asked for it.
    if design.shape == "refinement_pipeline":
        for it in range(1, max_iter + 1):
            if best_pass == len(visible):
                break
            f_state = GlobalState()
            f_state.add_message("user",
                f"PROBLEM:\n{problem.prompt}\n\n"
                f"PREVIOUS CODE:\n```python\n{current_code}\n```\n\n"
                f"TEST RESULTS ({best_pass}/{len(visible)}):\n{format_feedback(current_records)}\n\n"
                "Re-emit the COMPLETE corrected function."
            )
            revised = await fixer.run(f_state)
            new_pass, new_records = await evaluate(revised.code, visible, tool_factory)
            improved = "improved" if new_pass > best_pass else ("same" if new_pass == best_pass else "regressed")
            print(f"  ↪ iter {it} (fixer): {new_pass}/{len(visible)} visible — {improved}")
            if new_pass > best_pass:
                best_code = revised.code
                best_pass = new_pass
            current_code = revised.code
            current_records = new_records

    return best_code, best_pass, len(visible)


print(f"=== Running combo on {parse_roman.name} ===\n")
combo_code, visible_pass, visible_total = await run_combo(
    parse_roman, designer, coder, fixer, tool_factory
)

# Score against ALL tests (visible + hidden).
combo_score = score_candidate(combo_code, parse_roman)
print(f"\n=== Result ===")
print(f"baseline:  {result['passed']}/{result['total']}  ({result['passed']/result['total']:.0%})")
print(f"combo:     {combo_score['passed']}/{combo_score['total']}  ({combo_score['passed']/combo_score['total']:.0%})")
print(f"Δ:         {(combo_score['passed']-result['passed'])/result['total']:+.0%}")

=== Running combo on parse_roman ===



  ↪ designer picked: refinement_pipeline


  ↪ iter 0 (coder draft): 4/4 visible tests pass

=== Result ===
baseline:  14/14  (100%)
combo:     14/14  (100%)
Δ:         +0%


## 8. The full head-to-head

The numbers below come from `benchmarks/coding/run.py` averaged over **3 trials** on all 10 problems. Each trial runs both baseline and combo on the same problems with the same model and the same visible examples — only the architecture differs.

This is what makes the comparison fair: same model, same input. The only thing changing is the system around it.


In [8]:
# Pre-computed from 3 trials of benchmarks/coding/run.py
HEAD_TO_HEAD = {
    "model": "openrouter:openai/gpt-4o-mini",
    "trials": 3,
    "k_visible": 4,
    "max_iter": 3,
    "summary": {
        "baseline_pass1": {"mean": 0.73, "stdev": 0.06},
        "combo_pass1":    {"mean": 0.90, "stdev": 0.10},
        "baseline_rate":  {"mean": 0.80, "stdev": 0.06},
        "combo_rate":     {"mean": 0.94, "stdev": 0.09},
    },
    "per_problem": [
        ("parse_roman",                     0.55, 0.79),
        ("parse_csv_row",                   0.23, 0.73),
        ("text_justify",                    1.00, 1.00),
        ("min_window_substring",            1.00, 1.00),
        ("regex_match",                     1.00, 1.00),
        ("largest_rectangle_in_histogram",  1.00, 1.00),
        ("word_ladder_length",              0.38, 1.00),
        ("edit_distance",                   1.00, 1.00),
        ("subarray_sum_equals_k",           1.00, 1.00),
        ("palindrome_partition_min_cuts",   1.00, 1.00),
    ],
}

summary = HEAD_TO_HEAD["summary"]
print(f"Model: {HEAD_TO_HEAD['model']}  |  averaged over {HEAD_TO_HEAD['trials']} trials\n")
print(f"{'metric':<20s}  {'baseline':>16s}  {'combo':>16s}  {'Δ':>6s}")
for name, key_b, key_c in [
    ("pass@1",         "baseline_pass1", "combo_pass1"),
    ("test_pass_rate", "baseline_rate",  "combo_rate"),
]:
    b = summary[key_b]; c = summary[key_c]
    delta = c["mean"] - b["mean"]
    print(f"{name:<20s}  {b['mean']:>10.0%} (±{b['stdev']:.0%})  {c['mean']:>10.0%} (±{c['stdev']:.0%})  {delta:+5.0%}")

print(f"\nPer-problem breakdown (test_pass_rate, baseline → combo):")
print(f"{'problem':<35s} {'baseline':>10s} {'combo':>10s} {'Δ':>6s}  bar")
for pname, b_pct, c_pct in HEAD_TO_HEAD["per_problem"]:
    delta = c_pct - b_pct
    marker = "  ✓" if delta > 0.05 else ("  ✗" if delta < -0.05 else "   ")
    print(f"  {pname:<33s} {b_pct:>10.0%} {c_pct:>10.0%} {delta:+5.0%}{marker}")

Model: openrouter:openai/gpt-4o-mini  |  averaged over 3 trials

metric                        baseline             combo       Δ
pass@1                       73% (±6%)         90% (±10%)   +17%
test_pass_rate               80% (±6%)         94% (±9%)   +14%

Per-problem breakdown (test_pass_rate, baseline → combo):
problem                               baseline      combo      Δ  bar
  parse_roman                              55%        79%  +24%  ✓
  parse_csv_row                            23%        73%  +50%  ✓
  text_justify                            100%       100%   +0%   
  min_window_substring                    100%       100%   +0%   
  regex_match                             100%       100%   +0%   
  largest_rectangle_in_histogram          100%       100%   +0%   
  word_ladder_length                       38%       100%  +62%  ✓
  edit_distance                           100%       100%   +0%   
  subarray_sum_equals_k                   100%       100%   +0%   
  palindr

## 9. Where the wins come from

**Per-problem analysis:**

- **`parse_csv_row` (+50pp)** — RFC 4180 has many subtle rules. A single-shot baseline often gets quote-escaping wrong. The combo's first draft fails 1–3 visible tests, the fixer sees the failures, and the revision handles the escape rule correctly.
- **`word_ladder_length` (+62pp)** — BFS with one-letter-diff adjacency. Baseline sometimes is off-by-one on whether the start word counts. The visible tests immediately surface the bug.
- **`parse_roman` (+24pp)** — strict validation rules. Baseline often accepts `"IIII"` because it forgets the max-3-repeats rule. The visible test `parse_roman('IIII') == None` is the smoking gun; the fixer adds a validation pass.
- **6 problems at 100%/100%** — no room for the combo to improve. The combo correctly **does not regress**, which is what the keep-best policy guarantees.

**Why the architecture matters (not just "more LLM calls"):**

1. **Topology design** isolates the choice of shape from the work itself. A different domain (table QA, theorem proving, summarization) gets a different topology with **zero code changes**.
2. **Dynamic agent spawning via `AgentSpec`** keeps roles declarative. A future optimizer can mutate the coder and fixer prompts with GEPA (notebook 07) without recompiling Python.
3. **Dynamic tool spawning** is what enables the verifier loop without writing a custom test harness. The same `GeneratedToolSpec` + sandbox pattern works for math, data analysis, schema validation — anything you can run.

**Cost trade-off:**

- Baseline: 1 LLM call per problem, ~5s.
- Combo: 1 designer call + 1 coder call + up to 3 fixer calls + sandbox runs, ~15–50s.
- Net: 2–9× slower, 2–6× more LLM calls — for **+17pp pass@1** and **+14pp test_pass_rate**.

**Honest limits:**

- The loop only iterates when **visible** tests fail. Problems where both visible and hidden tests pass in one shot get no benefit (and pay no cost — the loop exits immediately).
- The combo cannot catch hidden-test failures it has no visible signal for. Increasing `k_visible` trades fairness for coverage.
- Keep-best only protects against **visible** regression. The fixer can still regress on hidden tests by fixing visible-test failures with brittle code. With `gpt-4o-mini` this is rare, and averaging trials washes it out.

**Where this points:**

- Run the same architecture with a stronger model. Gains should hold — the architecture wins are orthogonal to model quality.
- For domains with a real verifier (math proofs, SQL, executable code), the visible-test loop becomes nearly bulletproof.
- Combine with `OptimizationRunner` (notebook 07) to evolve the coder and fixer prompts over many problems.

---

**Reproduce the full head-to-head:**

```bash
python benchmarks/coding/run.py --trials 3 --model openrouter:openai/gpt-4o-mini
```


## Recap — the whole journey

Twelve notebooks. Here is the path:

| Notebook | Idea |
|---|---|
| 01 | An agent rates its own confidence. A watchdog stops the run if confidence drops. |
| 02 | A planner breaks a goal into subtasks. Each gets its own spawned agent. |
| 03 | Agents emit typed UI components instead of just text. |
| 04 | Compose agents with `Pipeline`, `Parallel`, `Router`, `RefinementLoop`. |
| 05 | One simple knob (`reasoning="high"`) controls deliberation across four providers. |
| 06 | GEPA evolves a single agent's prompt against a gold set. |
| 07 | GEPA evolves the planner inside an orchestrator. Same machinery, compound surface. |
| 08 | Evolve the **topology shape**, not just prompts. |
| 09 | Two-phase: topology first, then prompts on the winner. |
| 10 | Pick a topology **per request** at runtime. With caching. With decay. |
| 11 | Generate **new tools** at runtime and run them in a sandbox. |
| 12 | **All three** together on a real coding benchmark. +17pp pass@1. |

Each piece is independent. Each piece is small. They compose into systems that **design themselves**.

That's the whole library, in one sentence.
